In [1]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os
import geopandas as gpd
from libpysal.weights import KNN


In [2]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


In [ ]:
path_inputs = Path.cwd() / "inputs"
path_start_files = path_inputs / "input_files/C28"
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"

df_taz = pd.DataFrame(iter(DBF(path_taz_file)))
df_taz = df_taz[["TAZID", "TAZID_V911", "CO_TAZID", "DISTLRG"]]

omx_hbw_folder = (
    r"../WF-TDM-Documentation/_large_files/v1000/v1000-C28/2b-hbwdestchoice/"
)


In [3]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
hts_hh_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.household"
).to_dataframe()
hts_person_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.person"
).to_dataframe()
hts_trip_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.trip"
).to_dataframe()
hts_veh_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.vehicle"
).to_dataframe()


### Observed 2023


In [67]:
hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="unknown",
)

income_lookup = income_lookup[["hh_id", "income"]]

# 1. Merge vehicle lookup
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")

# 2. Re-establish 0 vehicles for missing households (and strip decimals)
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype("Int64")

# 3. Merge income lookup
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# 4. Fill missing income with "unknown"
hts_trip_23_merge["income"] = hts_trip_23_merge["income"].fillna("unknown")

# 5. Calculate segment
# Since vehown is now always a number (0, 1, or 2),
# we only label 'unknown' if the income data is missing.
hts_trip_23_merge["segment"] = np.where(
    (hts_trip_23_merge["income"] == "unknown") | (hts_trip_23_merge["income"] == ""),
    "unknown",
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
)

# 6. Final table selection
hts_trip_23_final = hts_trip_23_merge[
    ["segment", "vehown", "income", "pTAZID", "aTAZID", "trip_weight"]
]

hts_trip_23_final


,segment,vehown,income,pTAZID,aTAZID,trip_weight
0,1veh_lo,1,lo,2979.0,2966.0,17.687357
1,2veh_hi,2,hi,2779.0,2983.0,53.679940
2,2veh_hi,2,hi,830.0,866.0,101.865387
3,unknown,2,unknown,3058.0,3041.0,0.000000
4,2veh_hi,2,hi,1909.0,1971.0,292.121640
...,...,...,...,...,...,...
14972,1veh_hi,1,hi,512.0,958.0,22.263372
14973,2veh_hi,2,hi,1792.0,1786.0,456.865792
14974,2veh_hi,2,hi,1684.0,1467.0,0.000000
14975,0veh_lo,0,lo,1124.0,1007.0,0.000000


### Observed 2012


In [66]:
hts_trip_12_merge = hts_trip_12.copy()
hts_trip_12_merge = hts_trip_12_merge[
    [
        "record_id",
        "hptripid",
        "password",
        "hhmemberid",
        "p_SA1_TAZID",
        "a_SA1_TAZID",
        "p_SUBAREA_v30",
        "a_SUBAREA_v30",
        "PURP7_t",
        "weight",
    ]
]


# filter to HBW
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_12_merge = hts_trip_12_merge.merge(
    df_taz[["TAZID_V911", "TAZID"]],
    how="left",
    left_on="p_SA1_TAZID",
    right_on="TAZID_V911",
)
hts_trip_12_merge = hts_trip_12_merge.drop(
    columns=["p_SA1_TAZID", "TAZID_V911"]
).rename(columns={"TAZID": "pTAZID"})
hts_trip_12_merge = hts_trip_12_merge.merge(
    df_taz[["TAZID_V911", "TAZID"]],
    how="left",
    left_on="a_SA1_TAZID",
    right_on="TAZID_V911",
)
hts_trip_12_merge = hts_trip_12_merge.drop(
    columns=["a_SA1_TAZID", "TAZID_V911"]
).rename(columns={"TAZID": "aTAZID"})

# fitler to WF subarea
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["p_SUBAREA_v30"] == 1]
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["a_SUBAREA_v30"] == 1]


# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_12.copy().groupby("password").size().reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["password", "vehown"]]

# income lookup table
income_lookup = hts_hh_12.copy()[["password", "hh_income_cat"]]
income_lookup["income"] = np.select(
    [
        income_lookup["hh_income_cat"].isin([1, 2]),
        income_lookup["hh_income_cat"].isin([3, 4, 5]),
    ],
    ["lo", "hi"],
    default="unknown",  # <-- Catch unclassified income instead of using ""
)
income_lookup = income_lookup[["password", "income"]]

# merge vehown and income to trip table
hts_trip_12_merge = hts_trip_12_merge.merge(vehown_lookup, how="left", on="password")

hts_trip_12_merge["vehown"] = hts_trip_12_merge["vehown"].fillna(0).astype("Int64")

hts_trip_12_merge = hts_trip_12_merge.merge(income_lookup, how="left", on="password")

hts_trip_12_merge["income"] = hts_trip_12_merge["income"].fillna("unknown")

# 3. Corrected Segment Logic
hts_trip_12_merge["segment"] = np.where(
    (hts_trip_12_merge["income"] == "unknown") | (hts_trip_12_merge["income"] == ""),
    "unknown",  # Only unknown if income is missing
    hts_trip_12_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_12_merge["income"].astype(str),
)

hts_trip_12_final = hts_trip_12_merge[
    ["segment", "vehown", "income", "pTAZID", "aTAZID", "weight"]
]
hts_trip_12_final


,segment,vehown,income,pTAZID,aTAZID,weight
0,2veh_hi,2,hi,150.0,629.0,48.377805
1,2veh_hi,2,hi,29.0,977.0,63.424559
2,2veh_hi,2,hi,40.0,235.0,62.759109
3,2veh_hi,2,hi,40.0,235.0,62.759109
4,unknown,2,unknown,33.0,235.0,53.518451
...,...,...,...,...,...,...
10818,2veh_lo,2,lo,432.0,833.0,146.568901
10819,2veh_lo,2,lo,535.0,885.0,188.100313
10820,2veh_lo,2,lo,535.0,885.0,188.100313
10821,unknown,2,unknown,566.0,890.0,263.357977


### Merge Observed


In [68]:
# The keys we want to match on
merge_keys = ["segment", "vehown", "income", "pTAZID", "aTAZID"]

# 1. Aggregate 2012 trips down to unique combinations and sum the weights
hts_12_agg = (
    hts_trip_12_final.groupby(merge_keys, dropna=False)["weight"]
    .sum()
    .reset_index()
    .rename(columns={"weight": "trip_weight_12"})
)

# 2. Aggregate 2023 trips down to unique combinations and sum the weights
hts_23_agg = (
    hts_trip_23_final.groupby(merge_keys, dropna=False)["trip_weight"]
    .sum()
    .reset_index()
    .rename(columns={"trip_weight": "trip_weight_23"})
)

# 3. Outer merge the aggregated data
obs_merge = hts_12_agg.merge(
    hts_23_agg,
    how="outer",
    on=merge_keys,
)

# 4. Merge Origin District
obs_merge = obs_merge.merge(
    df_taz[["TAZID", "DISTLRG"]], how="left", left_on="pTAZID", right_on="TAZID"
)
obs_merge = obs_merge.rename(columns={"DISTLRG": "p_DISTLRG"}).drop(columns="TAZID")

# 5. Merge Destination District
obs_merge = obs_merge.merge(
    df_taz[["TAZID", "DISTLRG"]], how="left", left_on="aTAZID", right_on="TAZID"
)
obs_merge = obs_merge.rename(columns={"DISTLRG": "a_DISTLRG"}).drop(columns="TAZID")

# 6. Fill NaNs (Missing weights become 0)
obs_merge = obs_merge.fillna(0)

# 7. Final Column Ordering
obs_merge = obs_merge[
    [
        "segment",
        "vehown",
        "income",
        "pTAZID",
        "aTAZID",
        "p_DISTLRG",
        "a_DISTLRG",
        "trip_weight_12",
        "trip_weight_23",
    ]
]
obs_merge


,segment,vehown,income,pTAZID,aTAZID,p_DISTLRG,a_DISTLRG,trip_weight_12,trip_weight_23
0,0veh_hi,0,hi,377.0,426.0,5.0,5.0,187.790285,0.000000
1,0veh_hi,0,hi,440.0,437.0,5.0,5.0,214.664782,0.000000
2,0veh_hi,0,hi,883.0,1109.0,9.0,12.0,0.000000,326.911234
3,0veh_hi,0,hi,884.0,836.0,9.0,9.0,0.000000,64.448140
4,0veh_hi,0,hi,884.0,1536.0,9.0,14.0,0.000000,193.344420
...,...,...,...,...,...,...,...,...,...
11531,unknown,2,unknown,3432.0,2771.0,22.0,20.0,0.000000,745.705223
11532,unknown,2,unknown,3443.0,2646.0,22.0,20.0,155.057298,0.000000
11533,unknown,2,unknown,3443.0,3027.0,22.0,21.0,232.585947,0.000000
11534,unknown,2,unknown,3467.0,3497.0,22.0,22.0,147.614135,0.000000


In [69]:
hts_trip_12_hbw = hts_trip_12.copy()
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["PURP7_t"] == "HBW"]
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["p_SUBAREA_v30"] == 1]
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["a_SUBAREA_v30"] == 1]
print(hts_trip_12_hbw["weight"].sum())
print(obs_merge["trip_weight_12"].sum())

hts_trip_23_hbw = hts_trip_23.copy()
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["PURP7_t"] == "HBW"]
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["pSUBAREAID"] == 1]
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["aSUBAREAID"] == 1]
print(hts_trip_23_hbw["trip_weight"].sum())
print(obs_merge["trip_weight_23"].sum())


1228269.88801182
1228817.03028614
1711202.513165094
1711202.513165094


In [70]:
obs_merge.to_csv("observed_data_12_23.csv", index=False)


### Census Data


### Modeled Data


In [ ]:
# --------------------------------------------------------------------#
# Process Modeled Data (OMX Files)                                   #
# --------------------------------------------------------------------#
folder = "omx_hbw_folder"
files = [
    "pa_HBW_0veh_hi.omx",
    "pa_HBW_0veh_lo.omx",
    "pa_HBW_1veh_hi.omx",
    "pa_HBW_1veh_lo.omx",
    "pa_HBW_2veh_hi_noXI.omx",
    "pa_HBW_2veh_lo_noXI.omx",
]


def parse_veh_inc(fname):
    base = os.path.splitext(fname)[0]  # pa_HBW_2veh_hi_noXI
    base = base.replace("pa_", "")  # HBW_2veh_hi_noXI
    base = base.replace("_noXI", "")  # HBW_2veh_hi
    return base


dfs = []

for f in files:
    with omx.open_file(os.path.join(folder, f), "r") as omx_file:
        mat = omx_file["trips"][:]

    df = (
        pd.DataFrame(mat)
        .reset_index()
        .melt(id_vars="index", var_name="a_taz", value_name="trips")
        .rename(columns={"index": "p_taz"})
    )

    # shift from 0-based to 1-based TAZ
    df["p_taz"] += 1
    df["a_taz"] += 1

    df["veh_inc"] = parse_veh_inc(f)

    dfs.append(df)

mod_taz_hbw_0 = pd.concat(dfs, ignore_index=True)

# --------------------------------------------------------------------#
# Summarize Modeled Data                                             #
# --------------------------------------------------------------------#
# merge to get large districts and summarize accordingly
mod_taz_hbw_1 = mod_taz_hbw_0.merge(
    tazv10_wf, how="left", left_on="p_taz", right_on="TAZID"
)
mod_taz_hbw_1 = mod_taz_hbw_1.rename(columns={"DISTLRG": "p_DistLrg"}).drop(
    columns={"TAZID"}
)
mod_taz_hbw_2 = mod_taz_hbw_1.merge(
    tazv10_wf, how="left", left_on="a_taz", right_on="TAZID"
)
mod_taz_hbw_2 = mod_taz_hbw_2.rename(columns={"DISTLRG": "a_DistLrg"}).drop(
    columns={"TAZID"}
)

mod_dist_hbw_sum = (
    mod_taz_hbw_2[["veh_inc", "p_DistLrg", "a_DistLrg", "trips"]]
    .groupby(["veh_inc", "p_DistLrg", "a_DistLrg"], as_index=False)
    .agg(total_trips=("trips", "sum"))
)

# --------------------------------------------------------------------#
# Save Intermediate Files                                            #
# --------------------------------------------------------------------#
mod_dist_hbw_sum.to_csv("intermediate/mod_dist_hbw_sum_mc.csv", index=False)
